# Job Task: Feature Engineering

> **This notebook runs as a Databricks Job task.** It is Task 0 of the retrain pipeline.

Task 0 of the retrain pipeline:
- Read the raw source table (telco_churn or the combined training dataset)
- Write feature columns + customerID to the Unity Catalog Feature Store
- Create the feature table on first run; merge (upsert) on subsequent runs
- Pass the feature table name to downstream tasks via task values

In [ ]:
dbutils.widgets.text("catalog", "workshop", "Catalog")
dbutils.widgets.text("schema", "00_shared", "Schema")
dbutils.widgets.text("config_path", "config.yml", "Config Path")
dbutils.widgets.text("source_table", "", "Source Table (leave blank for default)")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
print(f"catalog={catalog}, schema={schema}")

In [ ]:
%pip install /Volumes/{catalog}/00_shared/wheels/churn_model-0.1.0-py3-none-any.whl databricks-feature-engineering --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
import yaml
from databricks.feature_engineering import FeatureEngineeringClient

catalog      = dbutils.widgets.get("catalog")
schema       = dbutils.widgets.get("schema")
config_path  = dbutils.widgets.get("config_path")
source_table_override = dbutils.widgets.get("source_table")

with open(config_path) as f:
    config = yaml.safe_load(f)

source_table = source_table_override if source_table_override else f"{catalog}.`00_shared`.telco_churn"
feature_table_name = f"{catalog}.{schema}.churn_features"

print(f"Source table  : {source_table}")
print(f"Feature table : {feature_table_name}")

# Select feature columns + the primary key (customerID)
feature_cols = (
    config["feature_columns"]["numeric"]
    + config["feature_columns"]["categorical"]
    + ["customerID"]
)

raw_df = spark.table(source_table)
features_df = raw_df.select(*feature_cols).dropDuplicates(["customerID"])
print(f"Feature rows  : {features_df.count():,}")

fe = FeatureEngineeringClient()

try:
    fe.create_table(
        name=feature_table_name,
        primary_keys=["customerID"],
        df=features_df,
        description="Telco churn features — source of truth for training and serving",
    )
    print(f"✓ Created feature table: {feature_table_name}")
except Exception as e:
    if "already exists" in str(e).lower():
        fe.write_table(name=feature_table_name, df=features_df, mode="merge")
        print(f"✓ Updated feature table: {feature_table_name}")
    else:
        raise

# Pass feature table name to downstream tasks
dbutils.jobs.taskValues.set("feature_table_name", feature_table_name)
print(f"\nTask value set: feature_table_name = {feature_table_name}")